# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL:

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata  # This is an object, not a dictionary
print(f"Dataset: {metadata.name}\nDescription: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

Entities are referenced by their `@id`.

**Note**: The Croissant schema may contain one or more record sets, each of which represents a logical table of data. We'll enumerate them:

In [ ]:
# List all record sets and their fields by @id
record_sets = metadata.record_sets
print(f"Number of record sets: {len(record_sets)}")
for rs in record_sets:
    print(f"RecordSet @id: {rs['@id']}")
    fields = rs.get('fields', [])
    print("  Fields:")
    for field in fields:
        print(f"    - {field['@id']} (name: {field.get('name')})")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

Use the record set and field `@id`s identified above.

We will extract all record sets into pandas DataFrames.

In [ ]:
# Extract available record set @id's
record_set_ids = [rs['@id'] for rs in metadata.record_sets]
print("RecordSets to extract:", record_set_ids)

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"DataFrame for {record_set_id} - columns: {list(df.columns)}")

# Show the head of the first record set as example
if record_set_ids:
    example_id = record_set_ids[0]
    print(f"\nSample records from RecordSet '{example_id}':")
    display(dataframes[example_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. 

**All field references should use their `@id`**.

*Below we automatically pick a numeric field for demonstration (if one exists).*

In [ ]:
import numpy as np

record_set_id = record_set_ids[0] if record_set_ids else None
df = dataframes[record_set_id]
fields = [f for f in metadata.record_sets[0].get('fields', [])] if record_set_id else []

# Try to find a numeric field by type
numeric_field_id = None
for field in fields:
    dtype = field.get('dataType', '').lower()
    if dtype in ['integer', 'float', 'number']:
        candidate = field['@id']
        if candidate in df.columns:  # Only use field that's present
            numeric_field_id = candidate
            break

if numeric_field_id is None:
    print("No numeric field found for demo. Please select manually.")
else:
    # Ensure numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    threshold = df[numeric_field_id].quantile(0.75) if not np.isnan(df[numeric_field_id]).all() else 0
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df.head())
    # Normalization
    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"\nNormalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    # Find group-by field
    group_field_id = None
    for field in fields:
        if field.get('dataType', '').lower() == 'text':
            candidate = field['@id']
            if candidate in df.columns and candidate != numeric_field_id:
                group_field_id = candidate
                break
    if group_field_id is not None:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().to_frame()
        print(f"\nGrouped data by {group_field_id} (mean {numeric_field_id}):")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

Below, we visualize the distribution of the numeric field (if available):

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and record_set_id:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()
    # If grouped, plot group means
    if group_field_id is not None:
        grouped = df.groupby(group_field_id)[numeric_field_id].mean().sort_values(ascending=False)
        plt.figure(figsize=(10, 5))
        sns.barplot(x=grouped.index.astype(str), y=grouped.values)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.xticks(rotation=60, ha='right')
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The dataset contains mass spectrometry records capturing protein characteristics from extracellular vesicles in stimulated human mast cells.
- Key fields and numeric values (such as molecular weight, peptide counts, or normalized abundances) were explored and visualized.
- The Croissant schema enables structured, reusable, programmatic exploration via Python and `mlcroissant`.
- For in-depth biological or scientific analysis, consult the complete field definitions and the cited article for research context.